In [2]:
import pandas as pd

# read file
gpm = pd.read_csv(
    "dataset/rainfall_gpm.csv",
    sep=",",
    quotechar='"',
    header=None,
    names=["Date", "Rainfall"]
)

gpm["Date"] = pd.to_datetime(gpm["Date"])
gpm["Rainfall"] = pd.to_numeric(gpm["Rainfall"])

print(gpm.head())
print(gpm.dtypes)

        Date  Rainfall
0 2015-01-01  0.021378
1 2015-01-02  0.000155
2 2015-01-03  0.000178
3 2015-01-04  0.000200
4 2015-01-05  0.000199
Date        datetime64[us]
Rainfall           float64
dtype: object


In [3]:
trmm = pd.read_csv(
    "dataset/rainfall_trmm.csv",
    header=None,
    names=["Date", "Rainfall"]
)

trmm["Date"] = pd.to_datetime(trmm["Date"])

print(trmm.head())
print(trmm.dtypes)

        Date  Rainfall
0 1998-01-01  0.000000
1 1998-01-02  0.010306
2 1998-01-03  0.023861
3 1998-01-04  0.000000
4 1998-01-05  0.000000
Date        datetime64[us]
Rainfall           float64
dtype: object


In [4]:
rainfall = pd.concat([trmm, gpm])

rainfall = rainfall.sort_values("Date")

print(rainfall.head())
print(rainfall.tail())

        Date  Rainfall
0 1998-01-01  0.000000
1 1998-01-02  0.010306
2 1998-01-03  0.023861
3 1998-01-04  0.000000
4 1998-01-05  0.000000
           Date   Rainfall
3921 2025-09-26   8.059369
3922 2025-09-27   6.688069
3923 2025-09-28  16.828831
3924 2025-09-29  24.443644
3925 2025-09-30   3.980065


In [5]:
sea = pd.read_csv(
    "dataset/sealevel_chennai.txt",
    sep=";",
    header=None
)

sea.columns = ["Decimal_Date", "Sea_Level", "Flag1", "Flag2"]

print(sea.head())

   Decimal_Date  Sea_Level  Flag1  Flag2
0     1916.0417       6897      0      0
1     1916.1250       6833      0      0
2     1916.2083       6806      0      0
3     1916.2917       6879      0      0
4     1916.3750       6979      0      0


In [6]:
import datetime

def decimal_to_date(decimal_year):
    year = int(decimal_year)
    remainder = decimal_year - year
    base = datetime.datetime(year, 1, 1)
    return base + datetime.timedelta(days=remainder * 365)

sea["Date"] = sea["Decimal_Date"].apply(decimal_to_date)
sea["Date"] = sea["Date"].dt.date
sea["Date"] = pd.to_datetime(sea["Date"])

print(sea.head())

   Decimal_Date  Sea_Level  Flag1  Flag2       Date
0     1916.0417       6897      0      0 1916-01-16
1     1916.1250       6833      0      0 1916-02-15
2     1916.2083       6806      0      0 1916-03-17
3     1916.2917       6879      0      0 1916-04-16
4     1916.3750       6979      0      0 1916-05-16


In [7]:
sea = sea[["Date", "Sea_Level"]]

print(sea.head())

        Date  Sea_Level
0 1916-01-16       6897
1 1916-02-15       6833
2 1916-03-17       6806
3 1916-04-16       6879
4 1916-05-16       6979


In [8]:
sea = sea[
    (sea["Date"] >= "1998-01-01") &
    (sea["Date"] <= "2023-12-31")
]
print(sea.head())
print(sea.tail())

          Date  Sea_Level
984 1998-01-16     -99999
985 1998-02-15     -99999
986 1998-03-18     -99999
987 1998-04-17     -99999
988 1998-05-17     -99999
           Date  Sea_Level
1290 2023-07-17       7142
1291 2023-08-17       7052
1292 2023-09-16       7069
1293 2023-10-16     -99999
1294 2023-11-16       7253


In [9]:
sea["Sea_Level"] = sea["Sea_Level"].replace(-99999, pd.NA)
print(sea.isna().sum())


Date          0
Sea_Level    40
dtype: int64


In [10]:
sea["Sea_Level"] = pd.to_numeric(sea["Sea_Level"], errors="coerce")
sea.loc[sea["Sea_Level"] == -99999, "Sea_Level"] = pd.NA
sea["Sea_Level"] = sea["Sea_Level"].interpolate()
print(sea.dtypes)
print(sea.isna().sum())

Date         datetime64[s]
Sea_Level          float64
dtype: object
Date          0
Sea_Level    12
dtype: int64


In [11]:
sea["Sea_Level"] = sea["Sea_Level"].interpolate()
sea["Sea_Level"] = sea["Sea_Level"].bfill()

In [12]:
print(sea.dtypes)
print(sea.isna().sum())

Date         datetime64[s]
Sea_Level          float64
dtype: object
Date         0
Sea_Level    0
dtype: int64


In [14]:
trmm_monthly = trmm.resample("ME", on="Date").sum().reset_index()

print(trmm_monthly.head())

        Date   Rainfall
0 1998-01-31   0.575627
1 1998-02-28   0.398774
2 1998-03-31   1.248927
3 1998-04-30  16.040662
4 1998-05-31  21.676685


In [15]:
gpm_monthly = gpm.resample("ME", on="Date").sum().reset_index()

print(gpm_monthly.head())

        Date   Rainfall
0 2015-01-31   1.323775
1 2015-02-28   0.021365
2 2015-03-31   2.148322
3 2015-04-30  57.940628
4 2015-05-31  55.521964


In [16]:
rainfall = pd.concat([trmm_monthly, gpm_monthly])
rainfall = rainfall.sort_values("Date")
print(rainfall.head())
print(rainfall.tail())

        Date   Rainfall
0 1998-01-31   0.575627
1 1998-02-28   0.398774
2 1998-03-31   1.248927
3 1998-04-30  16.040662
4 1998-05-31  21.676685
          Date    Rainfall
124 2025-05-31  113.007176
125 2025-06-30   66.905881
126 2025-07-31  119.189084
127 2025-08-31  158.236184
128 2025-09-30  172.596703


In [19]:
# sea = sea[["Date", "Sea_Level"]]
# data = pd.merge(
#     rainfall,
#     sea,
#     on="Date",
#     how="inner"
# )
#
# print(data.head())
# print(data.tail())
sea["Date"] = sea["Date"].dt.to_period("M").dt.to_timestamp("M")
print(sea.head())

          Date  Sea_Level
984 1998-01-31     7055.0
985 1998-02-28     7055.0
986 1998-03-31     7055.0
987 1998-04-30     7055.0
988 1998-05-31     7055.0


In [22]:

sea["Sea_Level"].nunique()

231

In [21]:
sea["Sea_Level"].describe()

count     311.000000
mean     7066.811897
std       126.433098
min      6769.000000
25%      6973.500000
50%      7052.000000
75%      7148.500000
max      7515.000000
Name: Sea_Level, dtype: float64

In [23]:
data = pd.merge(
    rainfall,
    sea,
    on="Date",
    how="inner"
)

print(data.head())
print(data.tail())
print(data.shape)

        Date   Rainfall  Sea_Level
0 1998-01-31   0.575627     7055.0
1 1998-02-28   0.398774     7055.0
2 1998-03-31   1.248927     7055.0
3 1998-04-30  16.040662     7055.0
4 1998-05-31  21.676685     7055.0
          Date    Rainfall  Sea_Level
366 2023-07-31  131.644348     7142.0
367 2023-08-31  145.624689     7052.0
368 2023-09-30  262.958837     7069.0
369 2023-10-31   43.040740     7161.0
370 2023-11-30  438.770541     7253.0
(371, 3)


In [25]:
soil = pd.read_csv(
    "dataset/soil_moisture.csv",
    header=None,
    names=["Date", "Soil_Moisture"]
)

print(soil.head())
print(soil.dtypes)

         Date  Soil_Moisture
0  1994-01-01       6.726681
1  1994-01-02       6.644792
2  1994-01-03       6.566135
3  1994-01-04       6.556434
4  1994-01-05       6.478244
Date                 str
Soil_Moisture    float64
dtype: object


In [26]:
soil["Date"] = pd.to_datetime(soil["Date"])
print(soil.dtypes)

Date             datetime64[us]
Soil_Moisture           float64
dtype: object


In [27]:
soil_monthly = soil.resample("ME", on="Date").mean().reset_index()

print(soil_monthly.head())

        Date  Soil_Moisture
0 1994-01-31       6.170056
1 1994-02-28       5.678075
2 1994-03-31       4.812068
3 1994-04-30       4.047286
4 1994-05-31       4.559653


In [28]:
data = pd.merge(
    data,
    soil_monthly,
    on="Date",
    how="inner"
)

print(data.head())
print(data.tail())
print(data.shape)

        Date   Rainfall  Sea_Level  Soil_Moisture
0 1998-01-31   0.575627     7055.0       6.895356
1 1998-02-28   0.398774     7055.0       6.017042
2 1998-03-31   1.248927     7055.0       5.282257
3 1998-04-30  16.040662     7055.0       4.622023
4 1998-05-31  21.676685     7055.0       4.327836
          Date    Rainfall  Sea_Level  Soil_Moisture
199 2014-08-31  135.538068     7035.0       5.167699
200 2014-09-30  121.575983     6961.0       5.310364
201 2014-10-31  295.714688     7073.0       6.118151
202 2014-11-30  120.631373     7091.0       6.147946
203 2014-12-31  120.849336     7109.0       5.402579
(204, 4)


In [30]:
X = data[["Rainfall", "Sea_Level", "Soil_Moisture"]]
y = data["Sea_Level"]

In [31]:
data["Rainfall_lag1"] = data["Rainfall"].shift(1)
data["Rainfall_3month_avg"] = data["Rainfall"].rolling(window=3).mean()
data["Sea_Level_trend"] = data["Sea_Level"].diff()
data = data.dropna()
print(data.head())
print(data.shape)

        Date   Rainfall  Sea_Level  Soil_Moisture  Rainfall_lag1  \
2 1998-03-31   1.248927     7055.0       5.282257       0.398774   
3 1998-04-30  16.040662     7055.0       4.622023       1.248927   
4 1998-05-31  21.676685     7055.0       4.327836      16.040662   
5 1998-06-30  28.041772     7055.0       3.988700      21.676685   
6 1998-07-31  84.962687     7055.0       4.983455      28.041772   

   Rainfall_3month_avg  Sea_Level_trend  
2             0.741110              0.0  
3             5.896121              0.0  
4            12.988758              0.0  
5            21.919706              0.0  
6            44.893715              0.0  
(202, 7)


In [33]:
data.to_csv("dataset/final_coastal_dataset.csv", index=False)